# Global Market Expansion and Localization Suite

## Taking the Day 5 Business Solution to the Next Level

### BUSINESS CHALLENGE:
Companies expanding into international markets need more than a generic brochure. They require tailored executive briefs targeted specifically at institutional investors versus prospective global customers, followed by immediate multi-language localization.

In this advanced community project, we extend Edward Donner's Day 5 multi-step agentic architecture into an automated global intelligence suite. 

### What this pipeline does:
1. **Intelligent Link Curation:** Uses an LLM to analyze raw webpage links and isolate strategic sub-pages (About, Products, Pricing, Careers) while stripping out noise like privacy policies.
2. **Multi-Page Data Aggregation:** Automatically crawls the landing page and all selected sub-pages to build a comprehensive corporate knowledge base.
3. **Dual-Persona Executive Synthesis:** Generates a structured executive brief customized for institutional investors and global enterprise clients.
4. **Multi-Language Localization:** Triggers secondary LLM translation calls to render the brief into fluent professional Spanish and French in real time using streaming animations.

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [ ]:
# load_dotenv(override=True)
# api_key = os.getenv("OPENROUTER_API_KEY")

# if api_key and api_key.startswith('sk-or-') and len(api_key) > 10:
#     print("OpenRouter API key loaded successfully.")
# else:
#     print("Please check your OPENROUTER_API_KEY in the .env file.")
    
# MODEL = 'openrouter/free' 
# openai = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=api_key
# )



load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()



In [ ]:
links = fetch_website_links("https://edwarddonner.com")
print(f"Total raw links found: {len(links)}")
links

## First Step: Filter Relevant Strategic Links in JSON Format

We use one-shot prompting to instruct the model to read the raw hyperlink list, ignore irrelevant links (such as privacy policies, terms of service, or email links), convert relative paths into full URLs, and return a structured JSON object.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a comprehensive corporate analysis,
such as links to an About page, Product page, Pricing page, or Careers/Jobs pages.
You should respond strictly in JSON format as in this example:

{
    "links": [
        {"type": "product page", "url": "https://full.url/goes/here/product" },
        {"type": "pricing page", "url": "https://full.url/pricing"}
    ]
}

"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""

Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a corporate evaluation, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, or social media links.

Links:
    
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt
    

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}...")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    
    # Extract only the JSON object between the first '{' and last '}'
    start_index = result.find("{")
    end_index = result.rfind("}")
    if start_index != -1 and end_index != -1:
        result = result[start_index : end_index + 1]
        
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Step 2: Multi-Page Content Aggregation

With the curated list of strategic URLs established, our pipeline loops through each link, fetches its text contents via our scraping utility, and combines them into a unified knowledge base representing the target company.

In [ ]:
def fetch_page_and_all_relevant_links(url):
    print(f"Fetching main landing page: {url}")
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        print(f"Fetching sub-page ({link['type']}): {link['url']}")
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    print("Finished aggregating all pages.")
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

## Step 3: Dual-Persona Executive Brief Synthesis

We now instruct the LLM to act as a senior market analyst, synthesizing the aggregated multi-page text into a professional executive brief. We implement streaming (`stream=True`) so the text renders with a typewriter effect in real time.

In [ ]:
brief_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a structured executive brief about the company for prospective customers, investors, and recruits.
Respond in clean markdown without code blocks.
Include dedicated sections for:
1. Executive Overview and Core Products
2. Customer Value Proposition
3. Investor Highlights and Market Position
4. Company Culture and Careers
"""

def get_brief_user_prompt(company_name, url):
    user_prompt = f"""
You are analyzing a company called: {company_name}
Below are the contents of its landing page and other relevant sub-pages.
Use this information to build a structured executive brief of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
print(get_brief_user_prompt("HuggingFace", "https://huggingface.co"))

In [ ]:
def stream_executive_brief(company_name, url):
    user_prompt = get_brief_user_prompt(company_name, url)
    print(f"Streaming executive brief for {company_name} using {MODEL}...")
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brief_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            update_display(Markdown(response), display_id=display_handle.display_id)
    return response

In [ ]:
english_brief = stream_executive_brief("HuggingFace", "https://huggingface.co")

## Step 4: Multi-Language Localization and Translation Extension

To satisfy the course challenge of adding secondary LLM calls, we take the generated English executive brief and pass it through a localization agent. This agent translates and adapts the document into fluent professional Spanish and French while preserving markdown structure.

In [ ]:
def stream_translated_brief(brief_text, target_language):
    print(f"Translating executive brief into {target_language} using {MODEL}...")
    translation_system_prompt = f"""
You are a professional corporate translator.
Translate the provided company executive brief into fluent, professional {target_language}.
Maintain all markdown formatting, headings, and bullet points without using code blocks.
"""
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": translation_system_prompt},
            {"role": "user", "content": f"Translate this executive brief into {target_language}:\n\n{brief_text}"}
        ],
        stream=True
    )
    translated_response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            translated_response += chunk.choices[0].delta.content
            update_display(Markdown(translated_response), display_id=display_handle.display_id)
    return translated_response

In [ ]:
spanish_brief = stream_translated_brief(english_brief, "Spanish")


In [ ]:
french_brief = stream_translated_brief(english_brief, "French")